In [ ]:
# 1. Mount Google Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

import os, joblib
model_load_dir = "/content/drive/MyDrive/model and Preprocessing Objects MLPA project"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def preprocess_and_engineer(df):
    # 0. Ensure all raw features exist
    for col in ['latitude','longitude','ndvi','soil_moisture_percent',
                'temperature_celsius','humidity_percent','rainfall_mm',
                'ph_level','nitrogen_kg_ha','phosphorus_kg_ha',
                'potassium_kg_ha','soil_fertility_index','crop_yield']:
        if col not in df.columns:
            df[col] = 0

    # 1. Numeric missing value imputation
    for col in df.select_dtypes(include='number').columns:
        df[col] = df[col].fillna(df[col].median())

    # 2. Categorical encoding
    for col, le in label_encoders.items():
        if col in df.columns:
            df[col] = df[col].fillna('Unknown')
            df[f'{col}_encoded'] = le.transform(df[col])

    # 3. Engineered features
    df['npk_total']     = df['nitrogen_kg_ha'] + df['phosphorus_kg_ha'] + df['potassium_kg_ha']
    df['n_to_p_ratio']  = df['nitrogen_kg_ha'] / (df['phosphorus_kg_ha'] + 1)
    df['n_to_k_ratio']  = df['nitrogen_kg_ha'] / (df['potassium_kg_ha'] + 1)
    df['p_to_k_ratio']  = df['phosphorus_kg_ha'] / (df['potassium_kg_ha'] + 1)

    df['temp_humidity_index']     = (df['temperature_celsius'] * df['humidity_percent']) / 100
    df['water_availability_index'] = (df['soil_moisture_percent']/100 + df['rainfall_mm']/300) / 2

    ph_score = 1 - abs(df['ph_level'] - 6.75)/6.75
    ph_score = ph_score.clip(0,1)
    df['soil_health_score'] = (df['soil_fertility_index']/100 + ph_score) / 2

    # 4. Scale numerical features
    num_cols = scaler.feature_names_in_ if hasattr(scaler, 'feature_names_in_') else df.select_dtypes(include='number').columns
    df[num_cols] = scaler.transform(df[num_cols])

    return df


In [ ]:
# 1. Load new data
import pandas as pd
from google.colab import files

new_data = pd.read_csv('/content/new_data_full.csv')

# 2. Preprocess and engineer features
df_proc = preprocess_and_engineer(new_data.copy())

# 3. Generate predictions for all objectives
new_data['water_stress_pred'] = dynamic_predict(water_model, df_proc, 'water_features')
new_data['disease_pred']      = dynamic_predict(disease_model, df_proc, 'disease_features')
new_data['soil_health_pred']  = dynamic_predict(soil_model, df_proc, 'soil_features')
new_data['irrigation_pred']   = dynamic_predict(irrigation_model, df_proc, 'irrigation_features')
new_data['nitrogen_pred']     = dynamic_predict(nitrogen_model, df_proc, 'nitrogen_features')
new_data['crop_yield_pred']   = dynamic_predict(yield_model, df_proc, 'yield_features')

# 4. Decode encoded predictions
new_data['water_stress_label'] = label_encoders['water_stress_level'].inverse_transform(new_data['water_stress_pred'])

# 5. Save and download results
new_data.to_csv('ai_agent_results.csv', index=False)
files.download('ai_agent_results.csv')

# 6. Display the results
new_data.head()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,latitude,longitude,ndvi,soil_moisture_percent,temperature_celsius,humidity_percent,rainfall_mm,ph_level,nitrogen_kg_ha,phosphorus_kg_ha,...,water_stress_level,disease_name,irrigation_needed,water_stress_pred,disease_pred,soil_health_pred,irrigation_pred,nitrogen_pred,crop_yield_pred,water_stress_label
0,12.9716,77.5946,0.45,28,30,55,10,6.8,80,30,...,Low,Healthy,No,2,1,-2.218548,1,70.059433,3.843502,Moderate


In [ ]:
import pandas as pd

def run_agent(input_data):
    """
    Run the full AI agent pipeline on input data.

    Args:
        input_data (str or pd.DataFrame): Path to CSV file or DataFrame with raw input features.

    Returns:
        pd.DataFrame: Original data with appended prediction columns.
    """

    # Load data if input is CSV path
    if isinstance(input_data, str):
        df = pd.read_csv(input_data)
    else:
        df = input_data.copy()

    # Preprocess and engineer features
    df_proc = preprocess_and_engineer(df.copy())

    # Predict on all objectives
    df['water_stress_pred'] = dynamic_predict(water_model, df_proc, 'water_features')
    df['disease_pred']      = dynamic_predict(disease_model, df_proc, 'disease_features')
    df['soil_health_pred']  = dynamic_predict(soil_model, df_proc, 'soil_features')
    df['irrigation_pred']   = dynamic_predict(irrigation_model, df_proc, 'irrigation_features')
    df['nitrogen_pred']     = dynamic_predict(nitrogen_model, df_proc, 'nitrogen_features')
    df['crop_yield_pred']   = dynamic_predict(yield_model, df_proc, 'yield_features')

    # Decode water stress labels for readability
    df['water_stress_label'] = label_encoders['water_stress_level'].inverse_transform(df['water_stress_pred'])

    return df

# Usage Example:
# result_df = run_agent("/content/new_data_full.csv")
# print(result_df.head())
# result_df.to_csv("ai_agent_results.csv", index=False)


In [ ]:
# Run your agent on input data (CSV path or DataFrame)
result_df = run_agent('/content/new_data_full.csv')  # or pass a pandas DataFrame

# Define the save path in your Google Drive
save_path = '/content/drive/MyDrive/ai_agent_results.csv'

# Save the results DataFrame to that path
result_df.to_csv(save_path, index=False)

print(f"Results saved to {save_path}")



Results saved to /content/drive/MyDrive/ai_agent_results.csv
